# L'assistant vocal bambara, au micro

Seulement la démo : aucun entraînement, aucune mesure. Elle charge
l'assemblage **en service** (`retenu.json` sur ton Drive, décidé par le
notebook de la phase 3 ; aujourd'hui Soloni + NLLB v1) et ouvre un lien
`gradio.live` pour parler à l'assistant depuis ton téléphone ou ton
ordinateur.

**Sans GPU, ça marche aussi** : Exécution → Modifier le type d'exécution →
**CPU**. Le quota de GPU n'est pas touché. Sur CPU, le LLM tourne dans sa
version allégée (GGUF, 1 Go, llama.cpp), comme pour le déploiement : compter
de 20 à 40 secondes par réponse, contre quelques secondes sur GPU. Les
salutations courantes répondent tout de suite (gabarits).

Exécution → **Tout exécuter**, autoriser Drive, puis ouvrir le lien affiché
par la dernière cellule (premier lancement : 5 à 10 minutes d'installation
et de téléchargement des modèles).

## 1. Installation

In [ ]:
BRANCH = "claude/chat-ia-bambara-dho8oa"

import os
os.chdir("/content")
if os.path.isdir("projet/.git"):
    !git -C projet pull -q
else:
    !git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd /content/projet
!pip install -q transformers sentencepiece sacrebleu soundfile librosa gradio

## 2. Les modèles en service

Lus dans `retenu.json` ; sans lui, ceux de la démo du 15 octobre (Whisper +
NLLB v1).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, subprocess
from pathlib import Path
import torch
from eval.promotion import lire_retenu

CKPT = "/content/drive/MyDrive/bambara/modeles"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EN_SERVICE = lire_retenu(f"{CKPT}/retenu.json", {
    "asr": {"model_id": f"{CKPT}/whisper-small-bm"}, "nom_asr": "Whisper",
    "bm2fr": f"{CKPT}/nllb-bm2fr", "nom_bm2fr": "NLLB v1",
    "fr2bm": f"{CKPT}/nllb-fr2bm", "nom_fr2bm": "NLLB v1"})
# La moyenne des poids v1/v2 vit sur le disque de Colab : refaite si elle est en service.
if EN_SERVICE["bm2fr"].startswith("/content/") and not Path(EN_SERVICE["bm2fr"], "model.safetensors").exists():
    subprocess.run(["python", "scripts/moyenne_poids.py", "--a", f"{CKPT}/nllb-bm2fr",
                    "--b", f"{CKPT}/nllb-bm2fr-v2", "--out", EN_SERVICE["bm2fr"]], check=True)

config = {"device": DEVICE, "asr": EN_SERVICE["asr"], "mt_in": {"model_id": EN_SERVICE["bm2fr"]},
          "mt_out": {"model_id": EN_SERVICE["fr2bm"]}}
if DEVICE == "cpu":
    # En pleine précision, le LLM prendrait 6 Go de mémoire (avec les deux NLLB,
    # plus que les 12 Go de Colab) et l'essentiel du temps : sa version GGUF
    # quantifiée (1 Go), via llama.cpp, comme pour le déploiement.
    from huggingface_hub import hf_hub_download
    gguf = hf_hub_download("bartowski/Qwen2.5-1.5B-Instruct-GGUF", "Qwen2.5-1.5B-Instruct-Q4_K_M.gguf")
    config["llm"] = {"backend": "llamacpp", "gguf_path": gguf}
CFG = "/content/assistant.json"
json.dump(config, open(CFG, "w"), indent=1)
print(f"En service : {EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']} ; "
      f"réponses : {EN_SERVICE['nom_fr2bm']} ; sur {DEVICE.upper()}")

In [ ]:
# Soloni et Soloba sont des modèles NeMo : quelques minutes d'installation.
if EN_SERVICE["asr"].get("kind") == "nemo":
    !pip install -q "nemo_toolkit[asr]"
# Sur CPU, llama.cpp pour le LLM (paquet précompilé).
if DEVICE == "cpu":
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

## 3. Au micro

Le lien public (`Running on public URL`) s'affiche ci-dessous. La cellule ne
rend pas la main : l'arrêter (■) ferme la démo.

In [ ]:
!python app/gradio_app.py --config $CFG --share